# YOLO26-S Baseline v3 — RSNA Pneumonia Detection
## Changes from v2: Excluded Not Normal class, imgsz=1024, AdamW optimizer, patience=30


---
## Step 1 — Install Dependencies
Run once, restart kernel before continuing.

In [ ]:
!pip install ultralytics pydicom opencv-python albumentations matplotlib seaborn pandas numpy tqdm scikit-learn Pillow pyyaml

---
## Step 2 — Imports

In [ ]:
import os, zipfile, shutil, warnings, time, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2, pydicom
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import albumentations as A
warnings.filterwarnings('ignore')
print('✅ All imports successful')

---
## Step 3 — Define All Paths

In [ ]:
# ── Edit these two lines if needed ──────────────────
ZIP_PATH = r"C:\Users\chait\Downloads\rsna-pneumonia-detection-challenge.zip"
BASE_DIR = Path(r"C:\Users\chait\Documents\rsna_pneumonia")
# ─────────────────────────────────────────────────────

RAW_DIR     = BASE_DIR / 'raw'
DICOM_DIR   = RAW_DIR  / 'stage_2_train_images'
PNG_DIR     = BASE_DIR / 'png_images'
PREPROC_DIR = BASE_DIR / 'preprocessed'
AUG_DIR     = BASE_DIR / 'augmented_v3'          # NEW — v3 clean dataset
LABEL_DIR   = BASE_DIR / 'yolo_labels_v3'         # NEW — v3 clean labels
YOLO_DIR    = BASE_DIR / 'yolo_dataset_v3'         # NEW — v3 clean dataset
RUNS_DIR    = BASE_DIR / 'runs'
PLOTS_DIR   = BASE_DIR / 'paper_plots_v3'          # NEW — v3 plots
MODELS_DIR  = BASE_DIR / 'models'

for d in [RAW_DIR, PNG_DIR, PREPROC_DIR, AUG_DIR, LABEL_DIR,
          RUNS_DIR, PLOTS_DIR, MODELS_DIR,
          YOLO_DIR/'images'/'train', YOLO_DIR/'images'/'val',
          YOLO_DIR/'labels'/'train', YOLO_DIR/'labels'/'val']:
    d.mkdir(parents=True, exist_ok=True)

print('All directories ready')
print(f'   Base  : {BASE_DIR}')
print(f'   YOLO  : {YOLO_DIR}')
print(f'   Plots : {PLOTS_DIR}')


---
## Step 4 — Extract ZIP

In [ ]:
if RAW_DIR.exists() and any(RAW_DIR.iterdir()):
    print('✅ Already extracted — skipping')
else:
    print('Extracting ZIP...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(RAW_DIR)
    print('✅ Done')

LABELS_CSV     = RAW_DIR / 'stage_2_train_labels.csv'
CLASS_INFO_CSV = RAW_DIR / 'stage_2_detailed_class_info.csv'
print(f'Labels CSV     : {LABELS_CSV.exists()}')
print(f'Class info CSV : {CLASS_INFO_CSV.exists()}')
print(f'DICOM folder   : {DICOM_DIR.exists()}')

---
## Step 5 — Load Labels

In [ ]:
df_labels = pd.read_csv(LABELS_CSV)
df_class  = pd.read_csv(CLASS_INFO_CSV)

# ── Exclude Not Normal patients immediately — before any processing ──────
# Not Normal = abnormal X-ray but NOT pneumonia (pleural effusion, masses etc.)
# Including them as negatives creates visual ambiguity during training
# and directly causes low precision and recall
valid_pids = df_class[
    df_class['class'] != 'No Lung Opacity / Not Normal'
]['patientId'].unique()

df_labels = df_labels[df_labels['patientId'].isin(valid_pids)].reset_index(drop=True)
df_class  = df_class[df_class['patientId'].isin(valid_pids)].reset_index(drop=True)

df_merged = df_labels.merge(
    df_class[['patientId','class']].drop_duplicates(), on='patientId', how='left')

print('After excluding Not Normal patients:')
print(f'Total rows     : {len(df_labels)}')
print(f'Unique patients: {df_labels["patientId"].nunique()}')
print(f'\nTarget dist  :\n{df_labels["Target"].value_counts()}')
print(f'\nClass dist   :\n{df_merged["class"].value_counts()}')


---
## Step 6A — Visualization: Class Distribution  *(Paper Figure 1)*

In [ ]:
img_class    = df_merged.groupby('patientId').agg(
    target=('Target','max'), cls=('class','first')).reset_index()
class_counts = img_class['cls'].value_counts()
colors       = ['#2ecc71','#e74c3c','#3498db']

fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle('RSNA Pneumonia Dataset — Class Distribution', fontsize=14, fontweight='bold')

bars = axes[0].bar(class_counts.index, class_counts.values,
                   color=colors[:len(class_counts)], edgecolor='black', linewidth=0.8)
axes[0].set_title('Sample Count per Class')
axes[0].set_ylabel('Number of Images')
axes[0].set_xlabel('Class')
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
                 str(val), ha='center', fontweight='bold')

axes[1].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%', colors=colors[:len(class_counts)],
            startangle=90, wedgeprops={'edgecolor':'black','linewidth':0.8})
axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig1_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Saved: fig1_class_distribution.png')

---
## Step 6B — Visualization: Bounding Box Statistics  *(Paper Figure 2)*

In [ ]:
df_pos         = df_labels[df_labels['Target']==1].copy()
df_pos['area'] = df_pos['width'] * df_pos['height']
df_pos['ar']   = df_pos['width'] / df_pos['height']

fig, axes = plt.subplots(2, 2, figsize=(14,10))
fig.suptitle('Bounding Box Statistics — Positive Cases', fontsize=14, fontweight='bold')

specs = [
    ('width',  'Width (px)',        '#3498db', 'red',    'mean'),
    ('height', 'Height (px)',       '#e74c3c', 'blue',   'mean'),
    ('area',   'Area (px²)',        '#9b59b6', 'orange', 'mean'),
    ('ar',     'Aspect Ratio (W/H)','#27ae60', 'red',    'one'),
]
for ax, (col, title, color, lc, kind) in zip(axes.flatten(), specs):
    ax.hist(df_pos[col], bins=50, color=color, edgecolor='black', linewidth=0.4)
    ref = df_pos[col].mean() if kind=='mean' else 1.0
    lbl = f'Mean: {ref:.0f}' if kind=='mean' else 'Square=1.0'
    ax.axvline(ref, color=lc, linestyle='--', label=lbl)
    ax.set_title(title); ax.set_xlabel(title); ax.set_ylabel('Count'); ax.legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig2_bbox_statistics.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Saved: fig2_bbox_statistics.png')

---
## Step 6C — Visualization: Opacity Location Heatmap  *(Paper Figure 3)*

In [ ]:
heatmap = np.zeros((1024,1024), dtype=np.float32)
for _, row in df_pos.iterrows():
    x1,y1 = max(0,int(row['x'])), max(0,int(row['y']))
    x2,y2 = min(1023,int(row['x']+row['width'])), min(1023,int(row['y']+row['height']))
    heatmap[y1:y2, x1:x2] += 1

fig, ax = plt.subplots(figsize=(8,8))
im = ax.imshow(heatmap, cmap='hot', interpolation='bilinear')
plt.colorbar(im, ax=ax, label='Annotation Frequency')
ax.set_title('Opacity Location Heatmap\n(All positive bounding boxes aggregated)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('X (px)'); ax.set_ylabel('Y (px)')
ax.axvline(512, color='cyan', linestyle='--', alpha=0.5, label='Centre')
ax.axhline(512, color='cyan', linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig3_opacity_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Saved: fig3_opacity_heatmap.png')

Step 6D — Visualization: Sample Annotated Images

In [ ]:
sample_pids = df_pos['patientId'].unique()[:8]
fig, axes   = plt.subplots(2, 4, figsize=(20,10))
fig.suptitle('Sample Chest X-Rays with Opacity Annotations', fontsize=14, fontweight='bold')

for ax, pid in zip(axes.flatten(), sample_pids):
    dcm_path = DICOM_DIR / f'{pid}.dcm'
    if not dcm_path.exists(): ax.axis('off'); continue
    ds  = pydicom.dcmread(str(dcm_path))
    img = ds.pixel_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255
    ax.imshow(img.astype(np.uint8), cmap='gray')
    bbs = df_labels[(df_labels['patientId']==pid)&(df_labels['Target']==1)]
    for _, r in bbs.iterrows():
        ax.add_patch(patches.Rectangle((r['x'],r['y']),r['width'],r['height'],
                                        lw=2, edgecolor='red', facecolor='none'))
    ax.set_title(pid[:12], fontsize=8); ax.axis('off')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig4_sample_annotations.png', dpi=300, bbox_inches='tight')
plt.show()
print(' Saved: fig4_sample_annotations.png')

---
## Step 7 Dicom TO PNG

In [ ]:
# Load class info to filter valid patients BEFORE converting DICOMs
# This saves disk space by skipping Not Normal patients entirely
LABELS_CSV     = RAW_DIR / 'stage_2_train_labels.csv'
CLASS_INFO_CSV = RAW_DIR / 'stage_2_detailed_class_info.csv'

df_class_filter = pd.read_csv(CLASS_INFO_CSV)
valid_pids = set(df_class_filter[
    df_class_filter['class'] != 'No Lung Opacity / Not Normal'
]['patientId'].unique())

print(f'Valid patients (Lung Opacity + Normal): {len(valid_pids)}')

dicom_files  = [f for f in DICOM_DIR.glob('*.dcm') if f.stem in valid_pids]
already_done = len(list(PNG_DIR.glob('*.png')))
print(f'DICOM: {len(dicom_files)} | PNGs done: {already_done}')

if already_done >= len(dicom_files):
    print('Skipping — all PNGs exist')
else:
    failed = []
    for dcm_path in tqdm(dicom_files, desc='DICOM→PNG'):
        out = PNG_DIR / (dcm_path.stem + '.png')
        if out.exists(): continue
        try:
            ds  = pydicom.dcmread(str(dcm_path))
            img = ds.pixel_array.astype(np.float32)
            img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255
            cv2.imwrite(str(out), img.astype(np.uint8))
        except: failed.append(dcm_path.name)
    print(f'Done — {len(dicom_files)-len(failed)} converted, {len(failed)} failed')


---
## Step 8A CLAHE (Offline augmentation ) Preview

In [ ]:
sample_png = list(PNG_DIR.glob('*.png'))[0]
raw_img    = cv2.imread(str(sample_png), cv2.IMREAD_GRAYSCALE)
clahe_img  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(raw_img)
z_img      = clahe_img.astype(np.float32)
z_img      = (z_img - z_img.mean()) / (z_img.std() + 1e-8)
z_img      = (z_img - z_img.min()) / (z_img.max() - z_img.min() + 1e-8) * 255

fig, axes = plt.subplots(1, 3, figsize=(18,5))
fig.suptitle('Preprocessing Pipeline Preview', fontsize=13, fontweight='bold')
for ax, img, title in zip(axes,
    [raw_img, clahe_img, z_img.astype(np.uint8)],
    ['Original', 'After CLAHE', 'After CLAHE + Z-score']):
    ax.imshow(img, cmap='gray'); ax.set_title(title); ax.axis('off')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig5_preprocessing_preview.png', dpi=300, bbox_inches='tight')
plt.show()
print(' Saved: fig5_preprocessing_preview.png')

---
## Step 8B — Image Preprocessing

In [ ]:
def preprocess_image(img_path, out_path):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None: return False
    img   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(img)
    img_f = img.astype(np.float32)
    img_f = (img_f - img_f.mean()) / (img_f.std() + 1e-8)
    img_f = (img_f - img_f.min())  / (img_f.max() - img_f.min() + 1e-8) * 255
    cv2.imwrite(str(out_path), cv2.cvtColor(img_f.astype(np.uint8), cv2.COLOR_GRAY2BGR))
    return True

png_files    = list(PNG_DIR.glob('*.png'))
already_done = len(list(PREPROC_DIR.glob('*.png')))
if already_done >= len(png_files):
    print(f' All {already_done} preprocessed images exist — skipping')
else:
    failed = []
    for p in tqdm(png_files, desc='Preprocessing'):
        if not (PREPROC_DIR/p.name).exists():
            if not preprocess_image(p, PREPROC_DIR/p.name): failed.append(p.name)
    print(f' Done — {len(png_files)-len(failed)} processed')

---
## Step 9 — SPLIT FIRST (BALANCE AND AUGMENT LATER)

In [ ]:
from sklearn.model_selection import train_test_split

# Get all unique patients with their targets — full unbalanced dataset
df_per_image = df_labels.groupby('patientId').agg(
    target=('Target','max')).reset_index()

all_pids    = df_per_image['patientId'].tolist()
all_targets = df_per_image['target'].tolist()

print(f'Total patients : {len(all_pids)}')
print(f'Positives      : {sum(all_targets)}')
print(f'Negatives      : {len(all_targets)-sum(all_targets)}')
print(f'Pos ratio      : {sum(all_targets)/len(all_targets):.3f}')

# Stratified split on full natural distribution
train_pids_all, val_pids = train_test_split(
    all_pids,
    test_size    = 0.2,
    random_state = 42,
    stratify     = all_targets
)

train_pids_all = set(train_pids_all)
val_pids       = set(val_pids)

# Verify no overlap
overlap = train_pids_all & val_pids
print(f'\nTrain patients : {len(train_pids_all)}')
print(f'Val patients   : {len(val_pids)}')
print(f'Patient overlap: {len(overlap)} -- {"PASS" if len(overlap)==0 else "FAIL"}')

# Val positive ratio should reflect natural distribution ~0.225
val_pos = df_per_image[
    (df_per_image['patientId'].isin(val_pids)) & 
    (df_per_image['target']==1)]
print(f'Val pos ratio  : {len(val_pos)/len(val_pids):.3f} (natural distribution)')

---
## Step 10 — Dataset Balancing
Undersample negatives to 1.5x positives — realistic slight imbalance for medical imaging.

In [ ]:
# Get train positives and negatives
train_df = df_per_image[df_per_image['patientId'].isin(train_pids_all)]
train_pos_pids = train_df[train_df['target']==1]['patientId'].tolist()
train_neg_pids = train_df[train_df['target']==0]['patientId'].tolist()

print(f'Train before balancing:')
print(f'  Positive : {len(train_pos_pids)}')
print(f'  Negative : {len(train_neg_pids)}')
print(f'  Ratio    : 1:{len(train_neg_pids)/len(train_pos_pids):.1f}')

# Undersample train negatives to 1.5x train positives
NEG_RATIO = 1.5
np.random.seed(42)
train_neg_sampled = list(np.random.choice(
    train_neg_pids,
    size    = int(len(train_pos_pids) * NEG_RATIO),
    replace = False
))

train_pids_balanced = set(train_pos_pids + train_neg_sampled)

print(f'\nTrain after balancing:')
print(f'  Positive : {len(train_pos_pids)}')
print(f'  Negative : {len(train_neg_sampled)}')
print(f'  Total    : {len(train_pids_balanced)}')
print(f'  Ratio    : 1:{len(train_neg_sampled)/len(train_pos_pids):.1f}')
print(f'\nVal unchanged: {len(val_pids)} patients (natural distribution)')
print(f'Patient overlap: {len(train_pids_balanced & val_pids)} -- {"PASS" if len(train_pids_balanced & val_pids)==0 else "FAIL"}')

---
## Step 10B — Define Offline Augmentation Pipeline
Medical-safe only — no hue/saturation for grayscale X-rays.

In [ ]:
aug_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.02, scale_limit=0.02, rotate_limit=3,
                       p=0.5, border_mode=cv2.BORDER_CONSTANT),
    A.RandomBrightnessContrast(brightness_limit=0.05, contrast_limit=0.05, p=0.3),
],
bbox_params=A.BboxParams(format='pascal_voc',
                          label_fields=['category_ids'], min_visibility=0.3))
print('Aug pipeline ready')

---
## Step 10C — Augment Train Positives Only, Val Natural:

In [ ]:
# Clear AUG_DIR
if AUG_DIR.exists():
    shutil.rmtree(str(AUG_DIR))
AUG_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE          = 1024
AUG_MULTIPLIER    = 1
all_image_records = []

# ── Train originals (balanced pos + neg) ─────────────────────────────────────
print('Copying train originals...')
for pid in tqdm(train_pids_balanced):
    src    = PREPROC_DIR / f'{pid}.png'
    dst    = AUG_DIR     / f'{pid}_orig.png'
    target = df_per_image[df_per_image['patientId']==pid]['target'].values[0]
    if not src.exists():
        continue
    if not dst.exists():
        shutil.copy2(str(src), str(dst))
    all_image_records.append({
        'patientId'  : pid,
        'file'       : dst.name,
        'target'     : int(target),
        'aug_bboxes' : None,
        'split'      : 'train'
    })

# ── Augment train positives only ─────────────────────────────────────────────
print('Augmenting train positives...')
for pid in tqdm(train_pos_pids):
    src = PREPROC_DIR / f'{pid}.png'
    if not src.exists():
        continue
    img = cv2.imread(str(src))
    if img is None:
        continue
    h, w   = img.shape[:2]
    bbs_df = df_labels[(df_labels['patientId']==pid) & (df_labels['Target']==1)]
    bboxes = [
        [max(0., float(r['x'])),
         max(0., float(r['y'])),
         min(float(w), float(r['x'] + r['width'])),
         min(float(h), float(r['y'] + r['height']))]
        for _, r in bbs_df.iterrows()
    ]
    bboxes = [b for b in bboxes if b[2] > b[0] and b[3] > b[1]]
    if not bboxes:
        continue
    for i in range(AUG_MULTIPLIER):
        out_name = f'{pid}_aug{i}.png'
        out_path = AUG_DIR / out_name
        try:
            res     = aug_pipeline(image=img, bboxes=bboxes,
                                   category_ids=[0]*len(bboxes))
            aug_bbs = res['bboxes']
            if not aug_bbs:
                continue
            if not out_path.exists():
                cv2.imwrite(str(out_path), res['image'])
            all_image_records.append({
                'patientId'  : pid,
                'file'       : out_name,
                'target'     : 1,
                'aug_bboxes' : aug_bbs,
                'split'      : 'train'
            })
        except:
            continue

# ── Val originals only — natural distribution, no augmentation ───────────────
print('Copying val originals...')
for pid in tqdm(val_pids):
    src    = PREPROC_DIR / f'{pid}.png'
    dst    = AUG_DIR     / f'{pid}_orig.png'
    target = df_per_image[df_per_image['patientId']==pid]['target'].values[0]
    if not src.exists():
        continue
    if not dst.exists():
        shutil.copy2(str(src), str(dst))
    all_image_records.append({
        'patientId'  : pid,
        'file'       : dst.name,
        'target'     : int(target),
        'aug_bboxes' : None,
        'split'      : 'val'
    })

# ── Summary & verification ────────────────────────────────────────────────────
df_all        = pd.DataFrame(all_image_records)
train_records = df_all[df_all['split']=='train']
val_records   = df_all[df_all['split']=='val']

print(f'\nDone')
print(f'  Train images : {len(train_records)}')
print(f'  Val images   : {len(val_records)}')
print(f'  Train pos    : {(train_records["target"]==1).sum()}')
print(f'  Train neg    : {(train_records["target"]==0).sum()}')
print(f'  Val pos      : {(val_records["target"]==1).sum()}')
print(f'  Val neg      : {(val_records["target"]==0).sum()}')
print(f'  Val pos ratio: {(val_records["target"]==1).sum()/len(val_records):.3f} (natural)')

# Leakage check
train_pids_check = set(train_records['patientId'])
val_pids_check   = set(val_records['patientId'])
overlap          = train_pids_check & val_pids_check
val_aug          = val_records[val_records['file'].str.contains('aug')]
print(f'\nLeakage checks:')
print(f'  Patient overlap    : {len(overlap)} -- {"PASS" if len(overlap)==0 else "FAIL"}')
print(f'  Aug images in val  : {len(val_aug)} -- {"PASS" if len(val_aug)==0 else "FAIL"}')

---
## Step 10D — Augmentation Preview

In [ ]:
spid = train_pos_pids[0]
fig, axes = plt.subplots(1, 2, figsize=(10,5))
fig.suptitle('Offline Augmentation — Train Positives Only',
             fontsize=13, fontweight='bold')

orig = AUG_DIR / f'{spid}_orig.png'
if orig.exists():
    img = cv2.cvtColor(cv2.imread(str(orig)), cv2.COLOR_BGR2RGB)
    axes[0].imshow(img)
    for _, r in df_labels[
        (df_labels['patientId']==spid) &
        (df_labels['Target']==1)].iterrows():
        axes[0].add_patch(patches.Rectangle(
            (r['x'],r['y']),r['width'],r['height'],
            lw=2, edgecolor='red', facecolor='none'))
    axes[0].set_title('Original (Train)')
    axes[0].axis('off')

ap = AUG_DIR / f'{spid}_aug0.png'
if ap.exists():
    axes[1].imshow(cv2.cvtColor(cv2.imread(str(ap)), cv2.COLOR_BGR2RGB))
    axes[1].set_title('Augmented #1 (Train only)')
    axes[1].axis('off')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig6_augmentation_preview.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig6_augmentation_preview.png')

---
## Step 11 — YOLO Label Conversion
Pixel (x,y,w,h) → YOLO normalised (cx,cy,w,h)

In [ ]:
IMG_SIZE  = 1024
LABEL_DIR = BASE_DIR / 'labels'
LABEL_DIR.mkdir(exist_ok=True)

def make_yolo_label(pid):
    rows  = df_labels[
        (df_labels['patientId']==pid) &
        (df_labels['Target']==1)]
    lines = []
    for _, r in rows.iterrows():
        cx = (r['x'] + r['width'] /2) / IMG_SIZE
        cy = (r['y'] + r['height']/2) / IMG_SIZE
        nw = r['width']  / IMG_SIZE
        nh = r['height'] / IMG_SIZE
        cx,cy = min(max(cx,0),1), min(max(cy,0),1)
        nw,nh = min(max(nw,0),1), min(max(nh,0),1)
        lines.append(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
    return '\n'.join(lines)

for _, rec in tqdm(df_all.iterrows(), total=len(df_all), desc='Labels'):
    stem = Path(rec['file']).stem
    lp   = LABEL_DIR / f'{stem}.txt'
    if rec['target'] == 0:
        lp.write_text('')
    elif rec['aug_bboxes'] is not None:
        lines = []
        for x1,y1,x2,y2 in rec['aug_bboxes']:
            cx = ((x1+x2)/2) / IMG_SIZE
            cy = ((y1+y2)/2) / IMG_SIZE
            nw = (x2-x1)     / IMG_SIZE
            nh = (y2-y1)     / IMG_SIZE
            cx,cy = min(max(cx,0),1), min(max(cy,0),1)
            nw,nh = min(max(nw,0),1), min(max(nh,0),1)
            lines.append(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        lp.write_text('\n'.join(lines))
    else:
        lp.write_text(make_yolo_label(rec['patientId']))

print(f'Labels: {len(list(LABEL_DIR.glob("*.txt")))} files')

---
## Step 12 UILD yOLO Dataset

In [ ]:
# Clear and rebuild yolo_dataset completely
if YOLO_DIR.exists():
    shutil.rmtree(str(YOLO_DIR))

for d in [YOLO_DIR/'images'/'train', YOLO_DIR/'images'/'val',
          YOLO_DIR/'labels'/'train', YOLO_DIR/'labels'/'val']:
    Path(d).mkdir(parents=True, exist_ok=True)

def copy_split(split):
    img_dst     = YOLO_DIR / 'images' / split
    lbl_dst     = YOLO_DIR / 'labels' / split
    split_recs  = df_all[df_all['split']==split]
    for _, rec in tqdm(split_recs.iterrows(),
                       total=len(split_recs),
                       desc=f'Copying {split}'):
        f       = rec['file']
        stem    = Path(f).stem
        src_img = AUG_DIR   / f
        src_lbl = LABEL_DIR / f'{stem}.txt'
        dst_img = img_dst   / f
        dst_lbl = lbl_dst   / f'{stem}.txt'
        if src_img.exists() and not dst_img.exists():
            shutil.copy2(str(src_img), str(dst_img))
        if src_lbl.exists() and not dst_lbl.exists():
            shutil.copy2(str(src_lbl), str(dst_lbl))

copy_split('train')
copy_split('val')

n_train = len(list((YOLO_DIR/'images'/'train').glob('*.png')))
n_val   = len(list((YOLO_DIR/'images'/'val').glob('*.png')))
print(f'YOLO dataset ready')
print(f'  Train: {n_train}')
print(f'  Val  : {n_val}')

---
## Step 13 — Create data.yaml

In [ ]:
data_yaml = {
    'path' : str(YOLO_DIR).replace('\\','/'),
    'train': 'images/train',
    'val'  : 'images/val',
    'nc'   : 1,
    'names': ['opacity']
}
yaml_path = YOLO_DIR / 'data.yaml'
with open(yaml_path,'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)
print('data.yaml created')
print(yaml.dump(data_yaml))

---
## Step 14A — Load YOLO26-S Model

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26s.pt')
print('YOLO26-S loaded')
model.info()

---
## Step 14B — Training Configuration

In [ ]:
TRAIN_CONFIG = dict(
    data            = str(yaml_path),
    epochs          = 75,
    imgsz           = 1024,
    batch           = 4,
    device          = 0,
    patience        = 30,
    project         = str(RUNS_DIR),
    name            = 'yolo26s_baseline_v3',
    exist_ok        = True,
    pretrained      = True,
    optimizer       = 'AdamW',
    lr0             = 0.001,
    lrf             = 0.01,
    weight_decay    = 0.0005,
    warmup_epochs   = 3.0,
    warmup_momentum = 0.8,
    warmup_bias_lr  = 0.1,
    seed            = 0,
    deterministic   = True,
    verbose         = True,
    save            = True,
    save_period     = 10,
    plots           = True,
    hsv_h           = 0.0,
    hsv_s           = 0.0,
    hsv_v           = 0.2,
    degrees         = 10,
    translate       = 0.05,
    scale           = 0.05,
    shear           = 0.0,
    perspective     = 0.0,
    flipud          = 0.0,
    fliplr          = 0.5,
    mosaic          = 0.0,
    mixup           = 0.0,
    copy_paste      = 0.0,
    close_mosaic    = 10,
    amp             = True,
    box             = 7.5,
    cls             = 0.5,
    dfl             = 1.5,
    nbs             = 64,
)
print('Config ready')
for k,v in TRAIN_CONFIG.items(): print(f'   {k:20s}: {v}')


---
## Step 14C — START TRAINING ▶
Real-time epoch output prints below automatically.

In [ ]:
print('='*60)
print('  YOLO26-S BASELINE v3 — TRAINING STARTED')
print('  Changes: No Not Normal class, imgsz=1024, AdamW, patience=30')
print('='*60)

results = model.train(**TRAIN_CONFIG)

print('='*60)
print('  TRAINING COMPLETE')
print('='*60)


---
## Step 15A — Load Best Weights & Validate

In [ ]:
best_weights = RUNS_DIR / 'yolo26s_baseline_v3' / 'weights' / 'best.pt'
print(f'Loading: {best_weights}')
best_model  = YOLO(str(best_weights))
val_metrics = best_model.val(data=str(yaml_path), imgsz=1024, device=0)
print('Validation complete')


---
## Step 15B — Extract & Display All Metrics

In [ ]:
map50   = val_metrics.box.map50
map75   = val_metrics.box.map75
map5095 = val_metrics.box.map
prec    = val_metrics.box.mp
rec     = val_metrics.box.mr
f1      = 2*(prec*rec)/(prec+rec+1e-8)
nparams = sum(p.numel() for p in best_model.model.parameters())/1e6

# FPS measurement on GPU
val_imgs_list = list((YOLO_DIR/'images'/'val').glob('*.png'))[:100]
t0  = time.time()
for ip in val_imgs_list: best_model.predict(str(ip), imgsz=1024, device=0, verbose=False)
fps = len(val_imgs_list)/(time.time()-t0)

print('\n'+'='*50)
print('  YOLO26-S BASELINE v3 — FINAL RESULTS')
print('='*50)
for name, val in [('mAP@0.5',map50),('mAP@0.75',map75),('mAP@0.5:0.95',map5095),
                  ('Precision',prec),('Recall',rec),('F1 Score',f1),
                  ('Params (M)',nparams),('FPS',fps)]:
    print(f'  {name:18s}: {val:.4f}')
print('='*50)

metrics_df = pd.DataFrame([{
    'Model':'YOLO26-S Baseline v3',
    'mAP@0.5':round(map50,4), 'mAP@0.75':round(map75,4),
    'mAP@0.5:0.95':round(map5095,4), 'Precision':round(prec,4),
    'Recall':round(rec,4), 'F1':round(f1,4),
    'Params(M)':round(nparams,2), 'FPS':round(fps,1)
}])
metrics_df.to_csv(BASE_DIR/'results_table_v3.csv', index=False)
print('\nSaved → results_table_v3.csv')
print(metrics_df.to_string(index=False))


---
## Step 16A — Loss Curves  *(Paper Figure 7)*

In [ ]:
df_res = pd.read_csv(RUNS_DIR/'yolo26s_baseline_v3'/'results.csv')
df_res.columns = df_res.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(18,5))
fig.suptitle('YOLO26-S Baseline v3 — Training & Validation Loss', fontsize=14, fontweight='bold')
for ax, (tc,vc,title) in zip(axes,[
    ('train/box_loss','val/box_loss','Box Loss'),
    ('train/cls_loss','val/cls_loss','Classification Loss'),
    ('train/dfl_loss','val/dfl_loss','DFL Loss')]):
    if tc in df_res.columns: ax.plot(df_res['epoch'],df_res[tc],label='Train',color='#3498db',lw=2)
    if vc in df_res.columns: ax.plot(df_res['epoch'],df_res[vc],label='Val',  color='#e74c3c',lw=2,ls='--')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR/'fig7_loss_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig7_loss_curves.png')


---
## Step 16B — mAP Curves  *(Paper Figure 8)*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle('YOLO26-S Baseline — mAP Progression', fontsize=14, fontweight='bold')
for ax, (col,label,color) in zip(axes,[
    ('metrics/mAP50(B)',   'mAP@0.5',      '#2ecc71'),
    ('metrics/mAP50-95(B)','mAP@0.5:0.95','#9b59b6')]):
    if col in df_res.columns:
        ax.plot(df_res['epoch'],df_res[col],color=color,lw=2)
        bi = df_res[col].idxmax()
        ax.axvline(df_res.loc[bi,'epoch'],color='red',ls='--',
                   alpha=0.7, label=f"Best: {df_res[col].max():.4f}")
    ax.set_title(label); ax.set_xlabel('Epoch'); ax.set_ylabel(label)
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR/'fig8_map_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Saved: fig8_map_curves.png')

---
## Step 16C — Precision & Recall Curves  *(Paper Figure 9)*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle('YOLO26-S Baseline — Precision & Recall Over Epochs', fontsize=14, fontweight='bold')
for ax, (col,label,color) in zip(axes,[
    ('metrics/precision(B)','Precision','#3498db'),
    ('metrics/recall(B)',   'Recall',   '#e74c3c')]):
    if col in df_res.columns: ax.plot(df_res['epoch'],df_res[col],color=color,lw=2)
    ax.set_title(label); ax.set_xlabel('Epoch'); ax.set_ylabel(label); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR/'fig9_precision_recall.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Saved: fig9_precision_recall.png')

---
## Step 16D — Copy Ultralytics Auto-Generated Plots

In [ ]:
run_dir = RUNS_DIR / 'yolo26s_baseline_v3'
for plot in ['confusion_matrix.png','confusion_matrix_normalized.png',
             'PR_curve.png','P_curve.png','R_curve.png','F1_curve.png','results.png']:
    src = run_dir / plot
    if src.exists():
        shutil.copy2(str(src), str(PLOTS_DIR/f'auto_{plot}'))
        print(f'Copied: {plot}')
    else:
        print(f'Not found: {plot}')


---
## Step 16E — Sample Predictions  *(Paper Figure 10)*

In [ ]:
val_samples = list((YOLO_DIR/'images'/'val').glob('*.png'))[:8]
fig, axes   = plt.subplots(2, 4, figsize=(20,10))
fig.suptitle('YOLO26-S v3 — Sample Predictions on Validation Set', fontsize=14, fontweight='bold')
for ax, ip in zip(axes.flatten(), val_samples):
    res = best_model.predict(str(ip), imgsz=1024, device=0, conf=0.25, verbose=False)[0]
    ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(ip.stem[:18], fontsize=8); ax.axis('off')
plt.tight_layout()
plt.savefig(PLOTS_DIR/'fig10_sample_predictions.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig10_sample_predictions.png')


---
## Step 17 — Save Model & Final Summary

In [ ]:
final_path = MODELS_DIR / 'yolo26s_baseline_v3_best.pt'
shutil.copy2(str(best_weights), str(final_path))

print('Model saved:', final_path)
print('\nPaper plots generated:')
for f in sorted(PLOTS_DIR.glob('*.png')): print(f'   {f.name}')
print('\nMetrics table: results_table_v3.csv')
print('\n'+'='*50)
print('  BASELINE v3 COMPLETE')
print('  Next: Run drop-neck v2 and drop-backbone v2 notebooks')
print('='*50)
